# ⚡ Chapter 3: Jupyter Productivity & Magic Commands

Master IPython's magic commands to supercharge your Jupyter workflow.

| # | Topic |
|---|-------|
| 1 | Line magics vs cell magics |
| 2 | `%timeit` and `%%timeit` for benchmarking |
| 3 | `%matplotlib inline` |
| 4 | `%store` for persisting variables |
| 5 | `%%writefile` to write cell contents to a file |
| 6 | `%%capture` to suppress output |
| 7 | `%env` for environment variables |
| 8 | `%debug` and `%pdb` for interactive debugging |
| 9 | `!pip install` from within notebooks |

---
## 3.1 Line Magics vs Cell Magics

IPython magics come in two flavours:

| Type | Prefix | Scope | Example |
|------|--------|-------|---------|
| **Line magic** | `%` | Operates on a single line | `%timeit sum(range(100))` |
| **Cell magic** | `%%` | Operates on the entire cell | `%%time` at the top of a cell |

### Automagic

IPython has **automagic** enabled by default — you can often omit the `%` for
line magics (e.g., just type `timeit sum(range(100))`). However, explicit `%`
is recommended for clarity.

In [ ]:
# List ALL available magics
%lsmagic

In [ ]:
# Get help on any magic
%timeit?

---
## 3.2 Benchmarking with `%timeit` and `%%timeit`

### `%timeit` — Line Magic
Times a **single statement** by running it many times automatically.

In [ ]:
# Compare list comprehension vs map
print('List comprehension:')
%timeit [x**2 for x in range(1000)]

print('\nmap() + list():')
%timeit list(map(lambda x: x**2, range(1000)))

print('\nGenerator expression (no list):')
%timeit sum(x**2 for x in range(1000))

In [ ]:
%%timeit
# %%timeit — Cell magic: times the ENTIRE cell
data = []
for i in range(1000):
    data.append(i ** 2)

In [ ]:
# Control the number of runs and loops
%timeit -n 100 -r 5 sorted(range(1000, 0, -1))

---
## 3.3 `%matplotlib inline`

This magic makes matplotlib plots appear directly in the notebook:

```python
%matplotlib inline
import matplotlib.pyplot as plt

plt.plot([1, 2, 3, 4], [1, 4, 9, 16])
plt.title('Simple Plot')
plt.show()
```

### Other backends

| Magic | Behavior |
|-------|----------|
| `%matplotlib inline` | Static images in the notebook |
| `%matplotlib notebook` | Interactive plots (zoom, pan) — classic notebook |
| `%matplotlib widget` | Interactive plots in JupyterLab |
| `%matplotlib svg` | SVG format (better quality for publications) |

In [ ]:
# Since matplotlib may not be installed, let's demonstrate
# the concept with a text-based "plot"

import math

def text_plot(func, start, end, width=60, height=15, title=''):
    """Create a simple text-based plot."""
    steps = width
    xs = [start + (end - start) * i / steps for i in range(steps + 1)]
    ys = [func(x) for x in xs]
    
    y_min, y_max = min(ys), max(ys)
    y_range = y_max - y_min or 1
    
    # Create grid
    grid = [[' ' for _ in range(width + 1)] for _ in range(height)]
    
    for i, y in enumerate(ys):
        row = int((y - y_min) / y_range * (height - 1))
        row = height - 1 - row  # flip y-axis
        if 0 <= row < height and 0 <= i <= width:
            grid[row][i] = '█'
    
    if title:
        print(f'\n  {title}')
    print(f'  {y_max:>8.2f} ┤', end='')
    print(''.join(grid[0]))
    for row in grid[1:-1]:
        print(f'           │', end='')
        print(''.join(row))
    print(f'  {y_min:>8.2f} ┤', end='')
    print(''.join(grid[-1]))
    print(f'           └{"─" * (width + 1)}')
    print(f'           {start:<8.1f}{" " * (width - 15)}{end:>8.1f}')


text_plot(math.sin, 0, 2 * math.pi, title='📈 sin(x) from 0 to 2π')
print()
text_plot(lambda x: x**2, -3, 3, title='📈 x² from -3 to 3')

---
## 3.4 `%store` — Persist Variables Between Notebooks

`%store` saves variables to IPython's database so they survive kernel restarts
and can be loaded in other notebooks.

```python
# Save a variable
my_data = [1, 2, 3, 4, 5]
%store my_data

# In another notebook or after kernel restart:
%store -r my_data     # restore it
print(my_data)        # → [1, 2, 3, 4, 5]
```

| Command | Action |
|---------|--------|
| `%store var` | Store variable |
| `%store -r var` | Restore variable |
| `%store -r` | Restore all stored variables |
| `%store -l` | List stored variables |
| `%store -d var` | Delete stored variable |
| `%store -z` | Delete all stored variables |

In [ ]:
# Demo: store a variable
course_name = 'Jupyter Fundamentals Course'
chapter_count = 5

%store course_name
%store chapter_count

# List stored variables
%store -l

---
## 3.5 `%%writefile` — Write Cell Contents to a File

Perfect for creating scripts, config files, or test data directly from a notebook.

In [ ]:
%%writefile hello_jupyter.py
#!/usr/bin/env python3
"""A simple script created with %%writefile magic."""

def greet(name):
    return f'Hello, {name}! Welcome to Jupyter. 🎉'

if __name__ == '__main__':
    print(greet('Student'))

In [ ]:
# Now run the script we just created
%run hello_jupyter.py

In [ ]:
# The function is now available in our namespace!
print(greet('Jupyter Master'))

---
## 3.6 `%%capture` — Suppress or Capture Output

Use `%%capture` to silently run a cell and store its output.

In [ ]:
%%capture captured_output
# This output will be captured, not displayed
for i in range(5):
    print(f'Processing item {i}...')
print('Done!')

In [ ]:
# Now we can inspect the captured output
print('Captured stdout:')
print(captured_output.stdout)

print('Captured stderr:')
print(repr(captured_output.stderr))

---
## 3.7 `%env` — Environment Variables

Read and set environment variables without leaving the notebook.

In [ ]:
# View all environment variables (just show a few)
import os
env_vars = dict(os.environ)
important_vars = ['HOME', 'USER', 'PATH', 'SHELL', 'LANG']

for var in important_vars:
    value = env_vars.get(var, 'not set')
    # Truncate long values like PATH
    if len(value) > 60:
        value = value[:57] + '...'
    print(f'{var:10s} = {value}')

In [ ]:
# Set an environment variable
%env MY_PROJECT=jupyter_course

# Read it back with Python
import os
print(f'MY_PROJECT = {os.environ.get("MY_PROJECT", "not set")}')

---
## 3.8 `%debug` and `%pdb` — Interactive Debugging

### Post-mortem debugging with `%debug`

After an exception, run `%debug` in the next cell to enter an interactive
debugger (pdb) at the point of failure.

```python
# Cell 1: This raises an error
def divide(a, b):
    return a / b

divide(10, 0)  # ZeroDivisionError!
```

```python
# Cell 2: Debug the error
%debug
# Now you can inspect variables: type 'a', 'b', 'up', 'down', 'quit'
```

### Automatic debugging with `%pdb`

```python
%pdb on     # Enable automatic debugger on exceptions
%pdb off    # Disable
```

### Useful pdb Commands

| Command | Action |
|---------|--------|
| `p expr` | Print expression |
| `pp expr` | Pretty-print expression |
| `l` | List source code |
| `n` | Next line |
| `s` | Step into function |
| `c` | Continue execution |
| `u` / `d` | Move up/down the call stack |
| `q` | Quit debugger |

In [ ]:
# Demo: a function that will raise an error
def process_data(data):
    """Process a list of numbers."""
    total = sum(data)
    count = len(data)
    average = total / count  # Could fail if data is empty!
    return average

# This works fine
result = process_data([10, 20, 30])
print(f'Average: {result}')

# Uncomment the next line to trigger an error, then run %debug in the next cell
# process_data([])  # ZeroDivisionError!

---
## 3.9 Installing Packages with `!pip`

### Basic usage

```python
# Install a package
!pip install requests

# Install a specific version
!pip install pandas==2.0.0

# Install from requirements file
!pip install -r requirements.txt

# Upgrade a package
!pip install --upgrade pip
```

### ⚠️ Best Practice: Use `sys.executable`

To ensure you install into the **correct** Python environment:

```python
import sys
!{sys.executable} -m pip install requests
```

This guarantees the package goes into the same environment as the running kernel.

In [ ]:
# Show what pip thinks is installed (first 10 packages)
import subprocess
import sys

result = subprocess.run(
    [sys.executable, '-m', 'pip', 'list', '--format=columns'],
    capture_output=True, text=True
)

lines = result.stdout.strip().split('\n')
print(f'Installed packages (showing first 10 of {len(lines) - 2}):\n')
for line in lines[:12]:
    print(line)

---
## 📝 Chapter 3 Summary

| Magic | Purpose |
|-------|---------|
| `%timeit` / `%%timeit` | Benchmark code performance |
| `%matplotlib inline` | Display plots in the notebook |
| `%store` | Persist variables between sessions |
| `%%writefile` | Write cell contents to a file |
| `%%capture` | Capture / suppress output |
| `%env` | Read / set environment variables |
| `%debug` / `%pdb` | Interactive post-mortem debugging |
| `!pip install` | Install packages from within notebook |

**Next up →** Chapter 4: Data Analysis Workflow in Jupyter